In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer

In [2]:
import sys
import pandas as pd

sys.path.append("./cluster_anlys")

from script_entropy import (
    run_script_entropy_over_pca_dims,
    save_batch_summary_df,
    save_single_partition_outputs, 
    run_script_entropy_over_random_pca_dims
)

In [3]:
seed_list = [0, 42, 1000, 9999, 1813382118, 827307999, 1627694678, 1911784257]

In [13]:
from script_entropy import sanity_check_words
sanity_check_words()

,token_str,script
0,shoulder,Latin
1,shoulders,Latin
2,homb,Latin
3,Shoulder,Latin
4,肩,Han
5,Schulter,Latin
6,плеч,Cyrillic


In [4]:
from script_entropy import toy_partition_test
toy_partition_test()

{'token_df':    cluster_id  token_str    script
 0           0   shoulder     Latin
 1           0  shoulders     Latin
 2           0   Schulter     Latin
 3           1          肩       Han
 4           1       плеч  Cyrillic
 5           1          肩       Han,
 'global_script_set': ['Cyrillic', 'Latin', 'Han'],
 'cluster_entropy_df':    cluster_id  Cyrillic  Latin  Han         H   H_norm
 0           0         0      3    0  0.000000  0.00000
 1           1         1      0    2  0.636514  0.57938,
 'partition_stats': {'num_clusters': 2,
  'mean_H': 0.3182570841474064,
  'std_H': 0.3182570841474064,
  'mean_H_norm': 0.28969008214284747,
  'std_H_norm': 0.28969008214284747}}

# Mistral-7B

In [12]:
# =========================================================
# params
# =========================================================
out_root = "comp"
model_name = "mistralai/Mistral-7B-v0.1"
space_name = "output_proj"

# ===== Step 1: Load tokenizer =====
tokenizer_path = f"{model_name}/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")


l2_norm = True
min_cluster_size = 5
min_samples = 5
metric = "euclidean"
cluster_selection_method = "eom"
cluster_selection_epsilon = 0.0

token_col = "token_str"
cluster_id_col = "cluster_id"

pca_dim_list = [5, 142, 997, 2084, 3031, 3459, 3938, 4088, 4096]

[✓] Tokenizer loaded from: mistralai/Mistral-7B-v0.1/tokenizer


In [6]:
batch_result = run_script_entropy_over_pca_dims(
    out_root=out_root,
    model_name=model_name,
    space_name=space_name,
    pca_dim_list=pca_dim_list,
    l2_norm=l2_norm,
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    metric=metric,
    cluster_selection_method=cluster_selection_method,
    cluster_selection_epsilon=cluster_selection_epsilon,
    tokenizer=tokenizer,
    print_columns_first_only=True,
)


=== Running script entropy: pca_dim=5 ===
cluster_df.columns = ['token_id', 'cluster_id', 'probability']

=== Running script entropy: pca_dim=142 ===

=== Running script entropy: pca_dim=997 ===

=== Running script entropy: pca_dim=2084 ===

=== Running script entropy: pca_dim=3031 ===

=== Running script entropy: pca_dim=3459 ===

=== Running script entropy: pca_dim=3938 ===

=== Running script entropy: pca_dim=4088 ===

=== Running script entropy: pca_dim=4096 ===


In [7]:
script_entropy_summary_df = batch_result["summary_df"]
script_entropy_summary_df

,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,num_clusters,mean_H,std_H,mean_H_norm,std_H_norm
0,mistralai/Mistral-7B-v0.1,output_proj,5,True,5,5,euclidean,eom,0.0,308,0.324168,0.484239,0.095310,0.142373
1,mistralai/Mistral-7B-v0.1,output_proj,142,True,5,5,euclidean,eom,0.0,79,0.097668,0.302831,0.028716,0.089037
2,mistralai/Mistral-7B-v0.1,output_proj,997,True,5,5,euclidean,eom,0.0,647,0.125131,0.206236,0.036790,0.060636
3,mistralai/Mistral-7B-v0.1,output_proj,2084,True,5,5,euclidean,eom,0.0,883,0.133825,0.201236,0.039347,0.059166
4,mistralai/Mistral-7B-v0.1,output_proj,3031,True,5,5,euclidean,eom,0.0,939,0.138824,0.201267,0.040816,0.059175
5,mistralai/Mistral-7B-v0.1,output_proj,3459,True,5,5,euclidean,eom,0.0,953,0.136805,0.196442,0.040223,0.057757
6,mistralai/Mistral-7B-v0.1,output_proj,3938,True,5,5,euclidean,eom,0.0,968,0.130007,0.191372,0.038224,0.056266
7,mistralai/Mistral-7B-v0.1,output_proj,4088,True,5,5,euclidean,eom,0.0,971,0.132795,0.194859,0.039044,0.057291
8,mistralai/Mistral-7B-v0.1,output_proj,4096,True,5,5,euclidean,eom,0.0,983,0.134932,0.195240,0.039672,0.057403


In [9]:
script_entropy_summary_df = batch_result["summary_df"]

summary_outpath = (
    f"{out_root}/{model_name}/{space_name}/"
    "script_entropy_summary.csv"
)

script_entropy_summary_df.to_csv(summary_outpath, index=False)
print(f"Saved batch summary -> {summary_outpath}")

Saved batch summary -> comp/mistralai/Mistral-7B-v0.1/output_proj/script_entropy_summary.csv


In [10]:
from script_entropy import save_single_partition_outputs

for pca_dim, result in batch_result["per_partition_results"].items():
    out_dir = (A
        f"{out_root}/{model_name}/{space_name}/"
        f"script_entropy/pca_{pca_dim}"
    )

    save_single_partition_outputs(
        result=result,
        out_dir=out_dir,
    )

Saved token-level output   -> comp/mistralai/Mistral-7B-v0.1/output_proj/script_entropy/pca_5/token_with_script.csv
Saved cluster-level output -> comp/mistralai/Mistral-7B-v0.1/output_proj/script_entropy/pca_5/cluster_script_entropy.csv
Saved summary output       -> comp/mistralai/Mistral-7B-v0.1/output_proj/script_entropy/pca_5/partition_script_entropy_summary.json
Saved token-level output   -> comp/mistralai/Mistral-7B-v0.1/output_proj/script_entropy/pca_142/token_with_script.csv
Saved cluster-level output -> comp/mistralai/Mistral-7B-v0.1/output_proj/script_entropy/pca_142/cluster_script_entropy.csv
Saved summary output       -> comp/mistralai/Mistral-7B-v0.1/output_proj/script_entropy/pca_142/partition_script_entropy_summary.json
Saved token-level output   -> comp/mistralai/Mistral-7B-v0.1/output_proj/script_entropy/pca_997/token_with_script.csv
Saved cluster-level output -> comp/mistralai/Mistral-7B-v0.1/output_proj/script_entropy/pca_997/cluster_script_entropy.csv
Saved summary o

### Random

In [13]:
script_entropy_summary_all = []

for pca_seed in seed_list:
    print(f"\n===== Running seed {pca_seed} =====")

    batch_result = run_script_entropy_over_random_pca_dims(
        out_root=out_root,
        model_name=model_name,
        space_name=space_name,
        pca_seed=pca_seed,
        pca_dim_list=pca_dim_list,
        l2_norm=l2_norm,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon,
        tokenizer=tokenizer,
        print_columns_first_only=True,
    )

    script_entropy_summary_df = batch_result["summary_df"]
    script_entropy_summary_all.append(script_entropy_summary_df)


    for pca_dim, result in batch_result["per_partition_results"].items():
        out_dir = (
            f"{out_root}/{model_name}/{space_name}/"
            f"random/seed_{pca_seed}/"
            f"script_entropy/pca_{pca_dim}"
        )

        save_single_partition_outputs(
            result=result,
            out_dir=out_dir,
        )

    summary_outpath = (
        f"{out_root}/{model_name}/{space_name}/"
        f"random/seed_{pca_seed}/"
        f"summary_random_pca_seed_{pca_seed}_script_entropy.csv"
    )

    script_entropy_summary_df.to_csv(summary_outpath, index=False)
    print(f"Saved batch summary -> {summary_outpath}")


# （可选）合并所有 seed 的 summary
all_summary_df = pd.concat(script_entropy_summary_all, ignore_index=True)

all_summary_outpath = (
    f"{out_root}/{model_name}/{space_name}/"
    f"random/script_entropy_all_seeds.csv"
)

all_summary_df.to_csv(all_summary_outpath, index=False)
print(f"\nSaved merged summary -> {all_summary_outpath}")


===== Running seed 0 =====

=== Running random script entropy: seed=0, pca_dim=5 ===
cluster_df.columns = ['token_id', 'cluster_id', 'probability']

=== Running random script entropy: seed=0, pca_dim=142 ===

=== Running random script entropy: seed=0, pca_dim=997 ===

=== Running random script entropy: seed=0, pca_dim=2084 ===

=== Running random script entropy: seed=0, pca_dim=3031 ===

=== Running random script entropy: seed=0, pca_dim=3459 ===

=== Running random script entropy: seed=0, pca_dim=3938 ===

=== Running random script entropy: seed=0, pca_dim=4088 ===

=== Running random script entropy: seed=0, pca_dim=4096 ===
Saved token-level output   -> comp/mistralai/Mistral-7B-v0.1/output_proj/random/seed_0/script_entropy/pca_5/token_with_script.csv
Saved cluster-level output -> comp/mistralai/Mistral-7B-v0.1/output_proj/random/seed_0/script_entropy/pca_5/cluster_script_entropy.csv
Saved summary output       -> comp/mistralai/Mistral-7B-v0.1/output_proj/random/seed_0/script_entrop

# Mixtral-8x7B

In [15]:
# =========================================================
# params
# =========================================================
out_root = "comp"
model_name = "mistralai/Mixtral-8x7B-v0.1"
space_name = "output_proj"

# ===== Step 1: Load tokenizer =====
tokenizer_path = f"{model_name}/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")


l2_norm = True
min_cluster_size = 5
min_samples = 5
metric = "euclidean"
cluster_selection_method = "eom"
cluster_selection_epsilon = 0.0

token_col = "token_str"
cluster_id_col = "cluster_id"

pca_dim_list = [8, 158, 1111, 2156, 3052, 3457, 3957, 4093, 4096]

[✓] Tokenizer loaded from: mistralai/Mixtral-8x7B-v0.1/tokenizer


In [12]:
batch_result = run_script_entropy_over_pca_dims(
    out_root=out_root,
    model_name=model_name,
    space_name=space_name,
    pca_dim_list=pca_dim_list,
    l2_norm=l2_norm,
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    metric=metric,
    cluster_selection_method=cluster_selection_method,
    cluster_selection_epsilon=cluster_selection_epsilon,
    tokenizer=tokenizer,
    print_columns_first_only=True,
)


=== Running script entropy: pca_dim=8 ===
cluster_df.columns = ['token_id', 'cluster_id', 'probability']

=== Running script entropy: pca_dim=158 ===

=== Running script entropy: pca_dim=1111 ===

=== Running script entropy: pca_dim=2156 ===

=== Running script entropy: pca_dim=3052 ===

=== Running script entropy: pca_dim=3457 ===

=== Running script entropy: pca_dim=3957 ===

=== Running script entropy: pca_dim=4093 ===

=== Running script entropy: pca_dim=4096 ===


In [13]:
script_entropy_summary_df = batch_result["summary_df"]
script_entropy_summary_df

,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,num_clusters,mean_H,std_H,mean_H_norm,std_H_norm
0,mistralai/Mixtral-8x7B-v0.1,output_proj,8,True,5,5,euclidean,eom,0.0,87,0.141175,0.263875,0.041508,0.077583
1,mistralai/Mixtral-8x7B-v0.1,output_proj,158,True,5,5,euclidean,eom,0.0,228,0.061555,0.166997,0.018098,0.049099
2,mistralai/Mixtral-8x7B-v0.1,output_proj,1111,True,5,5,euclidean,eom,0.0,978,0.160668,0.208533,0.047239,0.061312
3,mistralai/Mixtral-8x7B-v0.1,output_proj,2156,True,5,5,euclidean,eom,0.0,1082,0.169532,0.212628,0.049845,0.062516
4,mistralai/Mixtral-8x7B-v0.1,output_proj,3052,True,5,5,euclidean,eom,0.0,1098,0.176889,0.212463,0.052008,0.062467
5,mistralai/Mixtral-8x7B-v0.1,output_proj,3457,True,5,5,euclidean,eom,0.0,1097,0.178269,0.210618,0.052414,0.061925
6,mistralai/Mixtral-8x7B-v0.1,output_proj,3957,True,5,5,euclidean,eom,0.0,1094,0.173795,0.207971,0.051098,0.061146
7,mistralai/Mixtral-8x7B-v0.1,output_proj,4093,True,5,5,euclidean,eom,0.0,1091,0.173509,0.207080,0.051014,0.060884
8,mistralai/Mixtral-8x7B-v0.1,output_proj,4096,True,5,5,euclidean,eom,0.0,1091,0.173502,0.207075,0.051012,0.060883


In [14]:
script_entropy_summary_df = batch_result["summary_df"]

summary_outpath = (
    f"{out_root}/{model_name}/{space_name}/"
    "script_entropy_summary.csv"
)

script_entropy_summary_df.to_csv(summary_outpath, index=False)
print(f"Saved batch summary -> {summary_outpath}")

Saved batch summary -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/script_entropy_summary.csv


In [15]:
from script_entropy import save_single_partition_outputs

for pca_dim, result in batch_result["per_partition_results"].items():
    out_dir = (
        f"{out_root}/{model_name}/{space_name}/"
        f"script_entropy/pca_{pca_dim}"
    )

    save_single_partition_outputs(
        result=result,
        out_dir=out_dir,
    )

Saved token-level output   -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/script_entropy/pca_8/token_with_script.csv
Saved cluster-level output -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/script_entropy/pca_8/cluster_script_entropy.csv
Saved summary output       -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/script_entropy/pca_8/partition_script_entropy_summary.json
Saved token-level output   -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/script_entropy/pca_158/token_with_script.csv
Saved cluster-level output -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/script_entropy/pca_158/cluster_script_entropy.csv
Saved summary output       -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/script_entropy/pca_158/partition_script_entropy_summary.json
Saved token-level output   -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/script_entropy/pca_1111/token_with_script.csv
Saved cluster-level output -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/script_entropy/pca_1111/cluster_script_entropy.c

### random

In [16]:
script_entropy_summary_all = []

for pca_seed in seed_list:
    print(f"\n===== Running seed {pca_seed} =====")

    batch_result = run_script_entropy_over_random_pca_dims(
        out_root=out_root,
        model_name=model_name,
        space_name=space_name,
        pca_seed=pca_seed,
        pca_dim_list=pca_dim_list,
        l2_norm=l2_norm,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon,
        tokenizer=tokenizer,
        print_columns_first_only=True,
    )

    script_entropy_summary_df = batch_result["summary_df"]
    script_entropy_summary_all.append(script_entropy_summary_df)


    for pca_dim, result in batch_result["per_partition_results"].items():
        out_dir = (
            f"{out_root}/{model_name}/{space_name}/"
            f"random/seed_{pca_seed}/"
            f"script_entropy/pca_{pca_dim}"
        )

        save_single_partition_outputs(
            result=result,
            out_dir=out_dir,
        )

    summary_outpath = (
        f"{out_root}/{model_name}/{space_name}/"
        f"random/seed_{pca_seed}/"
        f"summary_random_pca_seed_{pca_seed}_script_entropy.csv"
    )

    script_entropy_summary_df.to_csv(summary_outpath, index=False)
    print(f"Saved batch summary -> {summary_outpath}")


# （可选）合并所有 seed 的 summary
all_summary_df = pd.concat(script_entropy_summary_all, ignore_index=True)

all_summary_outpath = (
    f"{out_root}/{model_name}/{space_name}/"
    f"random/script_entropy_all_seeds.csv"
)

all_summary_df.to_csv(all_summary_outpath, index=False)
print(f"\nSaved merged summary -> {all_summary_outpath}")


===== Running seed 0 =====

=== Running random script entropy: seed=0, pca_dim=8 ===
cluster_df.columns = ['token_id', 'cluster_id', 'probability']

=== Running random script entropy: seed=0, pca_dim=158 ===

=== Running random script entropy: seed=0, pca_dim=1111 ===

=== Running random script entropy: seed=0, pca_dim=2156 ===

=== Running random script entropy: seed=0, pca_dim=3052 ===

=== Running random script entropy: seed=0, pca_dim=3457 ===

=== Running random script entropy: seed=0, pca_dim=3957 ===

=== Running random script entropy: seed=0, pca_dim=4093 ===

=== Running random script entropy: seed=0, pca_dim=4096 ===
Saved token-level output   -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/random/seed_0/script_entropy/pca_8/token_with_script.csv
Saved cluster-level output -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/random/seed_0/script_entropy/pca_8/cluster_script_entropy.csv
Saved summary output       -> comp/mistralai/Mixtral-8x7B-v0.1/output_proj/random/seed_0/script

# gpt-oss-20B

In [4]:
# =========================================================
# params
# =========================================================
out_root = "comp"
model_name = "gpt-oss"
space_name = "output_proj"

# ===== Step 1: Load tokenizer =====
tokenizer_path = f"{model_name}/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")


l2_norm = True
min_cluster_size = 5
min_samples = 5
metric = "euclidean"
cluster_selection_method = "eom"
cluster_selection_epsilon = 0.0

token_col = "token_str"
cluster_id_col = "cluster_id"

pca_dim_list = [6, 182, 466, 739, 1591, 2264, 2532, 2868, 2880]

[✓] Tokenizer loaded from: gpt-oss/tokenizer


In [5]:
batch_result = run_script_entropy_over_pca_dims(
    out_root=out_root,
    model_name=model_name,
    space_name=space_name,
    pca_dim_list=pca_dim_list,
    l2_norm=l2_norm,
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    metric=metric,
    cluster_selection_method=cluster_selection_method,
    cluster_selection_epsilon=cluster_selection_epsilon,
    tokenizer=tokenizer,
    print_columns_first_only=True,
)


=== Running script entropy: pca_dim=6 ===
cluster_df.columns = ['token_id', 'cluster_id', 'probability']

=== Running script entropy: pca_dim=182 ===

=== Running script entropy: pca_dim=466 ===

=== Running script entropy: pca_dim=739 ===

=== Running script entropy: pca_dim=1591 ===

=== Running script entropy: pca_dim=2264 ===

=== Running script entropy: pca_dim=2532 ===

=== Running script entropy: pca_dim=2868 ===

=== Running script entropy: pca_dim=2880 ===


In [6]:
script_entropy_summary_df = batch_result["summary_df"]
script_entropy_summary_df

,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,num_clusters,mean_H,std_H,mean_H_norm,std_H_norm
0,gpt-oss,output_proj,6,True,5,5,euclidean,eom,0.0,1203,0.284744,0.478609,0.084562,0.142134
1,gpt-oss,output_proj,182,True,5,5,euclidean,eom,0.0,740,0.040971,0.151474,0.012167,0.044984
2,gpt-oss,output_proj,466,True,5,5,euclidean,eom,0.0,1941,0.127915,0.284627,0.037987,0.084527
3,gpt-oss,output_proj,739,True,5,5,euclidean,eom,0.0,2783,0.179682,0.335961,0.053361,0.099772
4,gpt-oss,output_proj,1591,True,5,5,euclidean,eom,0.0,3625,0.258150,0.393735,0.076664,0.116929
5,gpt-oss,output_proj,2264,True,5,5,euclidean,eom,0.0,3819,0.295526,0.429582,0.087764,0.127575
6,gpt-oss,output_proj,2532,True,5,5,euclidean,eom,0.0,3856,0.302609,0.437982,0.089867,0.130069
7,gpt-oss,output_proj,2868,True,5,5,euclidean,eom,0.0,3880,0.310559,0.447996,0.092228,0.133043
8,gpt-oss,output_proj,2880,True,5,5,euclidean,eom,0.0,3880,0.310515,0.448038,0.092215,0.133056


In [7]:
script_entropy_summary_df = batch_result["summary_df"]

summary_outpath = (
    f"{out_root}/{model_name}/{space_name}/"
    "script_entropy_summary.csv"
)

script_entropy_summary_df.to_csv(summary_outpath, index=False)
print(f"Saved batch summary -> {summary_outpath}")

Saved batch summary -> comp/gpt-oss/output_proj/script_entropy_summary.csv


In [8]:
from script_entropy import save_single_partition_outputs

for pca_dim, result in batch_result["per_partition_results"].items():
    out_dir = (
        f"{out_root}/{model_name}/{space_name}/"
        f"script_entropy/pca_{pca_dim}"
    )

    save_single_partition_outputs(
        result=result,
        out_dir=out_dir,
    )

Saved cluster-level output -> comp/gpt-oss/output_proj/script_entropy/pca_6/cluster_script_entropy.csv
Saved summary output       -> comp/gpt-oss/output_proj/script_entropy/pca_6/partition_script_entropy_summary.json
Saved cluster-level output -> comp/gpt-oss/output_proj/script_entropy/pca_182/cluster_script_entropy.csv
Saved summary output       -> comp/gpt-oss/output_proj/script_entropy/pca_182/partition_script_entropy_summary.json
Saved cluster-level output -> comp/gpt-oss/output_proj/script_entropy/pca_466/cluster_script_entropy.csv
Saved summary output       -> comp/gpt-oss/output_proj/script_entropy/pca_466/partition_script_entropy_summary.json
Saved cluster-level output -> comp/gpt-oss/output_proj/script_entropy/pca_739/cluster_script_entropy.csv
Saved summary output       -> comp/gpt-oss/output_proj/script_entropy/pca_739/partition_script_entropy_summary.json
Saved cluster-level output -> comp/gpt-oss/output_proj/script_entropy/pca_1591/cluster_script_entropy.csv
Saved summary 

In [9]:
script_entropy_summary_all = []

for pca_seed in seed_list:
    print(f"\n===== Running seed {pca_seed} =====")

    batch_result = run_script_entropy_over_random_pca_dims(
        out_root=out_root,
        model_name=model_name,
        space_name=space_name,
        pca_seed=pca_seed,
        pca_dim_list=pca_dim_list,
        l2_norm=l2_norm,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon,
        tokenizer=tokenizer,
        print_columns_first_only=True,
    )

    script_entropy_summary_df = batch_result["summary_df"]
    script_entropy_summary_all.append(script_entropy_summary_df)


    for pca_dim, result in batch_result["per_partition_results"].items():
        out_dir = (
            f"{out_root}/{model_name}/{space_name}/"
            f"random/seed_{pca_seed}/"
            f"script_entropy/pca_{pca_dim}"
        )

        save_single_partition_outputs(
            result=result,
            out_dir=out_dir,
        )

    summary_outpath = (
        f"{out_root}/{model_name}/{space_name}/"
        f"random/seed_{pca_seed}/"
        f"summary_random_pca_seed_{pca_seed}_script_entropy.csv"
    )

    script_entropy_summary_df.to_csv(summary_outpath, index=False)
    print(f"Saved batch summary -> {summary_outpath}")


# （可选）合并所有 seed 的 summary
all_summary_df = pd.concat(script_entropy_summary_all, ignore_index=True)

all_summary_outpath = (
    f"{out_root}/{model_name}/{space_name}/"
    f"random/script_entropy_all_seeds.csv"
)

all_summary_df.to_csv(all_summary_outpath, index=False)
print(f"\nSaved merged summary -> {all_summary_outpath}")


===== Running seed 0 =====

=== Running random script entropy: seed=0, pca_dim=6 ===
cluster_df.columns = ['token_id', 'cluster_id', 'probability']

=== Running random script entropy: seed=0, pca_dim=182 ===

=== Running random script entropy: seed=0, pca_dim=466 ===

=== Running random script entropy: seed=0, pca_dim=739 ===

=== Running random script entropy: seed=0, pca_dim=1591 ===

=== Running random script entropy: seed=0, pca_dim=2264 ===

=== Running random script entropy: seed=0, pca_dim=2532 ===

=== Running random script entropy: seed=0, pca_dim=2868 ===

=== Running random script entropy: seed=0, pca_dim=2880 ===
Saved cluster-level output -> comp/gpt-oss/output_proj/random/seed_0/script_entropy/pca_6/cluster_script_entropy.csv
Saved summary output       -> comp/gpt-oss/output_proj/random/seed_0/script_entropy/pca_6/partition_script_entropy_summary.json
Saved cluster-level output -> comp/gpt-oss/output_proj/random/seed_0/script_entropy/pca_182/cluster_script_entropy.csv
Sa